In [4]:
import urllib.request
import os
import pandas as pd
from datetime import datetime
import re
from io import StringIO

In [5]:
def download_vhi_data(save_dir='vhi_data'):
    os.makedirs(save_dir, exist_ok=True)
    base_url = 'https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean'
    
    existing = set()
    for fname in os.listdir(save_dir):
        m = re.match(r'vhi_province_(\d+)_', fname)
        if m:
            existing.add(int(m.group(1)))
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for pid in range(1, 28):
        if pid in existing:
            print(f'Провінція {pid:>2}: вже завантажено, пропускаємо.')
            continue
        url = base_url.format(pid=pid)
        filename = os.path.join(save_dir, f'vhi_province_{pid:02d}_{timestamp}.csv')
        try:
            urllib.request.urlretrieve(url, filename)
            print(f'Провінція {pid:>2}: завантажено → {filename}')
        except Exception as e:
            print(f'Провінція {pid:>2}: ПОМИЛКА — {e}')
    print('Завантаження завершено!')

download_vhi_data()

Провінція  1: вже завантажено, пропускаємо.
Провінція  2: вже завантажено, пропускаємо.
Провінція  3: вже завантажено, пропускаємо.
Провінція  4: вже завантажено, пропускаємо.
Провінція  5: вже завантажено, пропускаємо.
Провінція  6: вже завантажено, пропускаємо.
Провінція  7: вже завантажено, пропускаємо.
Провінція  8: вже завантажено, пропускаємо.
Провінція  9: вже завантажено, пропускаємо.
Провінція 10: вже завантажено, пропускаємо.
Провінція 11: вже завантажено, пропускаємо.
Провінція 12: вже завантажено, пропускаємо.
Провінція 13: вже завантажено, пропускаємо.
Провінція 14: вже завантажено, пропускаємо.
Провінція 15: вже завантажено, пропускаємо.
Провінція 16: вже завантажено, пропускаємо.
Провінція 17: вже завантажено, пропускаємо.
Провінція 18: вже завантажено, пропускаємо.
Провінція 19: вже завантажено, пропускаємо.
Провінція 20: вже завантажено, пропускаємо.
Провінція 21: вже завантажено, пропускаємо.
Провінція 22: вже завантажено, пропускаємо.
Провінція 23: вже завантажено, п

In [6]:
NOAA_TO_UA = {
    1:(23,'Черкаська'), 2:(24,'Чернігівська'), 3:(25,'Чернівецька'),
    4:(26,'Крим'), 5:(3,'Дніпропетровська'), 6:(4,'Донецька'),
    7:(8,'Івано-Франківська'), 8:(20,'Харківська'), 9:(21,'Херсонська'),
    10:(22,'Хмельницька'), 11:(9,'Київська'), 12:(10,'Київ (місто)'),
    13:(11,'Кіровоградська'), 14:(12,'Луганська'), 15:(13,'Львівська'),
    16:(14,'Миколаївська'), 17:(15,'Одеська'), 18:(16,'Полтавська'),
    19:(17,'Рівненська'), 20:(18,'Сумська'), 21:(19,'Тернопільська'),
    22:(1,'Вінницька'), 23:(2,'Волинська'), 24:(6,'Закарпатська'),
    25:(7,'Запорізька'), 26:(5,'Житомирська'), 27:(27,'Севастополь'),
}

In [7]:
def read_vhi_files(save_dir='vhi_data'):
    all_frames = []
    for fname in sorted(os.listdir(save_dir)):
        if not fname.endswith('.csv'):
            continue
        m = re.match(r'vhi_province_(\d+)_', fname)
        if not m:
            continue
        noaa_id = int(m.group(1))
        fpath = os.path.join(save_dir, fname)

        with open(fpath, 'r', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()

        data_lines = []
        header_found = False
        for line in lines:
            stripped = line.strip()
            if not header_found:
                if stripped.startswith('year'):
                    header_found = True
                    clean_header = stripped.replace('<br>', '').rstrip(',')
                    data_lines.append(clean_header)
            else:
                if stripped and not stripped.startswith('<'):
                    data_lines.append(stripped.rstrip(','))

        if not data_lines:
            continue

        df = pd.read_csv(StringIO('\n'.join(data_lines)), skipinitialspace=True)
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
        df.columns = df.columns.str.strip().str.replace('<br>', '', regex=False)
        keep = [c for c in ['year','week','SMN','SMT','VCI','TCI','VHI'] if c in df.columns]
        df = df[keep]
        df.replace(-1, pd.NA, inplace=True)
        for col in ['SMN','SMT','VCI','TCI','VHI']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df.dropna(subset=['year','week'], inplace=True)
        df['year'] = df['year'].astype(int)
        df['week'] = df['week'].astype(int)

        ua_index, ua_name = NOAA_TO_UA.get(noaa_id, (noaa_id, f'Province_{noaa_id}'))
        df['province_id'] = noaa_id
        df['ua_index'] = ua_index
        df['ua_name'] = ua_name
        all_frames.append(df)

    full_df = pd.concat(all_frames, ignore_index=True)
    print(f'Завантажено {len(all_frames)} областей, {len(full_df)} рядків.')
    return full_df

df = read_vhi_files()
df.head(10)

Завантажено 27 областей, 60345 рядків.


,year,week,SMN,SMT,VCI,TCI,VHI,province_id,ua_index,ua_name
0,1982,2,0.054,262.29,46.83,31.75,39.29,1,23,Черкаська
1,1982,3,0.055,263.82,48.13,27.24,37.68,1,23,Черкаська
2,1982,4,0.053,265.33,46.09,23.91,35.00,1,23,Черкаська
3,1982,5,0.050,265.66,41.46,26.65,34.06,1,23,Черкаська
4,1982,6,0.048,266.55,36.56,29.46,33.01,1,23,Черкаська
5,1982,7,0.048,267.84,32.17,31.14,31.65,1,23,Черкаська
6,1982,8,0.050,269.30,30.30,32.50,31.40,1,23,Черкаська
7,1982,9,0.052,270.75,28.23,35.22,31.73,1,23,Черкаська
8,1982,10,0.056,272.73,25.25,37.63,31.44,1,23,Черкаська
9,1982,11,0.069,275.46,26.81,34.79,30.80,1,23,Черкаська


In [14]:
def get_vhi_by_region_year(df, ua_index, year):
    result = df[(df['ua_index']==ua_index) & (df['year']==year)][['week','VHI','ua_name']]
    if result.empty:
        print(f'Даних немає для ua_index={ua_index}, рік={year}')
        return result
    print(f"VHI для {result['ua_name'].iloc[0]}, {year} рік:")
    return result[['week','VHI']].reset_index(drop=True)

# VHI для Вінницької за 2020 рік
print(get_vhi_by_region_year(df, ua_index=1, year=2020))

VHI для Вінницька, 2020 рік:
    week    VHI
0      1  41.75
1      2  43.90
2      3  45.12
3      4  45.32
4      5  44.78
5      6  43.67
6      7  43.46
7      8  44.86
8      9  46.62
9     10  48.32
10    11  47.55
11    12  47.81
12    13  46.68
13    14  43.64
14    15  40.71
15    16  40.20
16    17  42.54
17    18  45.15
18    19  49.85
19    20  56.29
20    21  59.87
21    22  62.94
22    23  65.05
23    24  67.03
24    25  67.26
25    26  66.86
26    27  66.67
27    28  67.17
28    29  68.55
29    30  68.47
30    31  66.98
31    32  64.96
32    33  63.07
33    34  61.97
34    35  61.14
35    36  58.19
36    37  54.14
37    38  50.12
38    39  47.17
39    40  47.27
40    41  46.01
41    42  44.15
42    43  42.12
43    44  40.37
44    45  42.85
45    46  42.44
46    47  42.73
47    48  44.57
48    49  45.67
49    50  43.89
50    51  42.81
51    52  44.23


In [11]:
def get_vhi_by_regions_years(df, ua_indices, year_start, year_end):
    result = df[(df['ua_index'].isin(ua_indices)) & (df['year']>=year_start) & (df['year']<=year_end)]
    result = result[['year','week','VHI','ua_name']].sort_values(['ua_name','year','week'])
    print(f'Знайдено {len(result)} записів.')
    return result.reset_index(drop=True)

#VHI для кількох областей за діапазон років
print(get_vhi_by_regions_years(df, ua_indices=[1, 20], year_start=2015, year_end=2020))

Знайдено 624 записів.
     year  week    VHI     ua_name
0    2015     1  47.82   Вінницька
1    2015     2  50.16   Вінницька
2    2015     3  50.53   Вінницька
3    2015     4  49.33   Вінницька
4    2015     5  46.84   Вінницька
..    ...   ...    ...         ...
619  2020    48  36.12  Харківська
620  2020    49  37.49  Харківська
621  2020    50  36.90  Харківська
622  2020    51  38.88  Харківська
623  2020    52  42.54  Харківська

[624 rows x 4 columns]


In [13]:
def get_vhi_stats(df, ua_indices, year_start=None, year_end=None):
    subset = df[df['ua_index'].isin(ua_indices)].copy()
    if year_start: subset = subset[subset['year']>=year_start]
    if year_end:   subset = subset[subset['year']<=year_end]
    return subset.groupby(['ua_index','ua_name'])['VHI'].agg(
        мінімум='min', максимум='max', середнє='mean', медіана='median'
    ).reset_index()
#  Статистика
print(get_vhi_stats(df, ua_indices=[1, 5, 20]))

   ua_index      ua_name  мінімум  максимум    середнє  медіана
0         1    Вінницька    19.45     77.71  47.831474    47.68
1         5  Житомирська    10.88     96.69  44.935478    43.03
2        20   Харківська     9.36     91.42  48.250371    47.37
